# Reading Data for What Is Missing

**DS4DH · Module 01 — Framing the Right Question**

*Technique:* Absence audit — missing variables, missing populations, missing granularity

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Sagaustus/ds4dh-colab-pack/blob/main/notebooks/01b_absence_audit.ipynb)

Data: `merged_dataset.csv` — from the `data/` folder of this pack.

---

In [ ]:
# Setup — run this first.
import os, warnings
warnings.filterwarnings('ignore')
import numpy as np
import pandas as pd

# This notebook reads the CSVs sitting next to it. In Colab, upload them from
# the pack's data/ folder when prompted. The exists() guard means a re-run
# part-way through a session will not ask you to upload all over again.
NEEDED = ['merged_dataset.csv']
missing = [f for f in NEEDED if not os.path.exists(f)]
if missing:
    try:
        from google.colab import files
        print('Upload from the data/ folder of the pack: ' + ', '.join(missing))
        files.upload()
    except ImportError:
        raise SystemExit('Place these next to the notebook: ' + ', '.join(missing))

df       = pd.read_csv('merged_dataset.csv')
CITIES = ['Montréal', 'Toronto', 'Edmonton', 'Vancouver']

print(f'Loaded. df has {len(df):,} rows and {df.shape[1]} columns.')

## What this notebook does

Most published errors in data analysis are not arithmetic. They are claims made
about things the dataset never measured.

There are three kinds of absence, and they fail differently:

1. **Missing variables** — the factor is not a column. No method recovers it.
2. **Missing populations** — the people are not rows. Your sample is not who you think.
3. **Missing granularity** — the unit is too coarse to see the pattern you care about.

This notebook audits all three, and ends with a written scope statement.

## Absence 1 — Variables that are not here

Read the column list and ask what a housing researcher would want that is absent.

In [ ]:
print('Columns present:')
for c in df.columns:
    print('   ', c)

print()
print('Not present, and not derivable from what is:')
for want in ['dwelling age / condition', 'household size', 'tenure length',
             'rent control status', 'time (this is one census snapshot)',
             'race / visible minority status', 'core housing need flag']:
    print('   ✗', want)

The absence of **time** is the consequential one. A single snapshot cannot
support any claim containing the words *rising*, *worsening*, *increasingly*, or
*trend*. Those words require at least two points.

The absence of **household size** matters differently: a 35% STIR means something
different for a single person than for a family of five, and nothing here
distinguishes them.

In [ ]:
# Absence 2 — populations. Who is counted, and who is not?
csd = df.dropna(subset=['csd_code'])
base = csd[(csd['immigrant_status'] == 'Total Immigrant Status')
           & (csd['cma'].isin(CITIES))].copy()

reported = base['Renter'].notna().sum()
print(f'CSDs in the four cities:            {len(base):>4}')
print(f'  with a reported renter STIR:      {reported:>4}')
print(f'  suppressed / not reported:        {len(base) - reported:>4}')
print()
print('Census suppression removes small-population cells. So the missing places')
print('are not a random subset — they are systematically the small ones.')

In [ ]:
# Is that systematic? Compare the population of reported vs suppressed CSDs.
have = base[base['Renter'].notna() & base['tot_pop'].notna()]
gone = base[base['Renter'].isna() & base['tot_pop'].notna()]

print(f'{"":<26}{"n":>5}{"median pop":>13}')
print('-' * 44)
print(f'{"Renter STIR reported":<26}{len(have):>5}{have["tot_pop"].median():>13,.0f}')
print(f'{"Renter STIR suppressed":<26}{len(gone):>5}{gone["tot_pop"].median():>13,.0f}')
print()
if len(gone) and have['tot_pop'].median() > gone['tot_pop'].median():
    print('Confirmed: suppressed CSDs are much smaller. Any statement you make')
    print('about "municipalities" is really about the larger ones.')

### 🔧 Your turn 1

Change the audit above from `Renter` to `Owner` and re-run.

Are the same places missing? If owner data survives where renter data does not,
what does that do to the renter–owner gap you can compute?

## Absence 3 — Granularity

The dataset has two levels: CSD and CMA. It does not have neighbourhoods.

A CSD can be a city of 2.6 million or a village of 600. Averaging within one
hides everything that varies inside it — the subject of the MAUP notebook (09b).

In [ ]:
pops = base.dropna(subset=['tot_pop'])['tot_pop']
print(f'CSD population — smallest: {pops.min():>12,.0f}')
print(f'                  median:  {pops.median():>12,.0f}')
print(f'                  largest: {pops.max():>12,.0f}')
print(f'                  ratio:   {pops.max() / pops.min():>12,.0f}x')
print()
print('One "observation" can be four orders of magnitude bigger than another.')
print('Every unweighted mean in this course treats them as equals.')

### 🔧 Your turn 2

Write the scope statement for this dataset in three sentences: what it can
support, what it cannot, and which absence you consider most limiting.

Keep it. Notebook 12 asks you to compare it against the one you write at the end.

<details markdown="1">
<summary><b>What you should have seen</b> — click to expand</summary>

**Your turn 1.** Owner STIR is suppressed less often than renter STIR, because
most small municipalities have more owner households than renter households.
The consequence is subtle and important: `renter_owner_gap` can only be computed
where *both* survive, so the gap is measured on a sample that is more urban and
larger-population than the full set of CSDs. The gap is real, but it is not a gap
"across Canadian municipalities" — it is a gap across the ones big enough to be
published.

**Your turn 2.** A defensible scope statement:

> These data describe shelter-cost-to-income ratios for 189 census subdivisions
> in four Canadian CMAs at a single census point. They support comparison of
> housing burden between owner and renter households, and between immigrant and
> non-immigrant households, within and across those four metropolitan areas.
> They cannot support any claim about change over time, about household-level
> circumstances, or about municipalities too small for census reporting. The most
> limiting absence is time: without a second point, nothing here distinguishes a
> housing crisis from a stable if unequal equilibrium.

</details>

## Where this stops

An absence audit is not a caveat you add at the end. It determines which of the
questions from 01a the dataset can answer at all — and it has just ruled out
every question containing the word "rising".

Next: **02a — Descriptive Statistics**, now that you know what you are describing.